In [1]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from IPython.display import display
import numpy as np
import plotly.graph_objects as go
import math

In [2]:
#### This code frame uses widgets in search of the grid for hyperbloid formfinding

In [3]:
############ customization
length = 24
width = 24
n1 = 3
n2 = 3
judge = 0
ratio = 0
Initial_r = 1

In [4]:
## grid generation
def generate_rectangular_grid(length, width, n1, n2=2, judge=0, z=0, height=0, d=0, Ratio = ratio, Initial_r = Initial_r):
    
    ratios = [1.0] * n1  

    mid_index = n1 // 2 

    for i in range(mid_index, n1):
        decay_factor = Ratio ** (i - mid_index)  
        ratios[i] *= (Initial_r + d * decay_factor)  
        ratios[n1 - 1 - i] *= (Initial_r + d * decay_factor)  

    total_ratio = sum(ratios)
#     print(ratios)
    normalized_ratios = [r / total_ratio for r in ratios]
#     print(normalized_ratios)
    x_points = [0.0] * (n1 + 1)  
    for i in range(1, n1 + 1):
        x_points[i] = x_points[i - 1] + normalized_ratios[i - 1] * length

    y_points = [j * (width / n2) for j in range(n2, -1, -1)]

    grid_points = []
    for x in x_points:
        for y in y_points:
            if y == width / 2:
                grid_points.append([x, y, height])
            else:
                grid_points.append([x, y, z])

    if judge == 1:
        corners = [
            [x_points[0], y_points[0], z],
            [x_points[0], y_points[-1], z],
            [x_points[-1], y_points[0], z],
            [x_points[-1], y_points[-1], z]
        ]
        grid_points = [point for point in grid_points if point not in corners]

    return grid_points

def plot_grid(grid_points, length, width):
    x = [point[0] for point in grid_points]
    y = [point[1] for point in grid_points]
    z = [point[2] for point in grid_points]

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(x, y, z, c='r', marker='o', s=50, label='Grid Points')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.legend()
    plt.show()

def generate_connectivity_matrix(new_coords):

    indexed_points = {tuple(point): idx + 1 for idx, point in enumerate(new_coords)}

    connectivity = []


    x_values = sorted(set(point[0] for point in new_coords))
    for x in x_values:

        points_on_line = [point for point in new_coords if point[0] == x]
        points_on_line.sort(key=lambda p: p[1], reverse=True) 


        for i in range(len(points_on_line) - 1):
            node1 = indexed_points[tuple(points_on_line[i])]
            node2 = indexed_points[tuple(points_on_line[i + 1])]
            connectivity.append([node1, node2])


    y_values = sorted(set(point[1] for point in new_coords))
    for y in y_values:
       
        points_on_line = [point for point in new_coords if point[1] == y]
        points_on_line.sort(key=lambda p: p[0])  


        for i in range(len(points_on_line) - 1):
            node1 = indexed_points[tuple(points_on_line[i])]
            node2 = indexed_points[tuple(points_on_line[i + 1])]
            connectivity.append([node1, node2])

    return connectivity

In [5]:
 def interactive_plot(d):
    grid_points = generate_rectangular_grid(length, width, n1, n2, judge, d=d)
    plot_grid(grid_points, length, width)

d_slider = widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05, description='Offset d:')
widgets.interact(interactive_plot, d=d_slider)

interactive(children=(FloatSlider(value=0.0, description='Offset d:', max=2.0, step=0.05), Output()), _dom_cla…

<function __main__.interactive_plot(d)>

In [6]:
grid_points = generate_rectangular_grid(length, width, n1, n2, judge, d=d_slider.value)
connectivity = generate_connectivity_matrix(grid_points)
# print(d_slider.value)
# print(grid_points)
print(connectivity)

[[1, 2], [2, 3], [3, 4], [5, 6], [6, 7], [7, 8], [9, 10], [10, 11], [11, 12], [13, 14], [14, 15], [15, 16], [4, 8], [8, 12], [12, 16], [3, 7], [7, 11], [11, 15], [2, 6], [6, 10], [10, 14], [1, 5], [5, 9], [9, 13]]


In [7]:
############# Problem context
n_dof_per_node = 6  # Degrees of freedom per node
total_dof = n_dof_per_node * len(grid_points)
grid_points = torch.tensor(grid_points)
########## Surrounding fixed
x_max = grid_points[:, 0].max()
x_min = grid_points[:, 0].min()
y_max = grid_points[:, 1].max()
y_min = grid_points[:, 1].min()
Fixed_nodes = torch.where(
#     (grid_points[:, 0] == x_max) |  # x = x_max
#     (grid_points[:, 0] == x_min) |  # x = x_min
    (grid_points[:, 1] == y_max) |  # y = y_max
    (grid_points[:, 1] == y_min)    # y = y_min
)[0]

Fixed_nodes += 1
Free_nodes = []
# n_elements = len(connectivity)
n_nodes = len(grid_points)
for i in range(1, n_nodes + 1):
    if i not in Fixed_nodes:
        Free_nodes.append(i)


        
        
f_n = Free_nodes
F_value = torch.tensor([-1.0] * len(Free_nodes)) * 1000 # The force value/direction
r = 1 / torch.max(F_value)


####### BCs
fixed_dof = []
for node in Fixed_nodes:
    fixed_dof.extend([(node - 1) * 6 + i for i in range(6)])

print(len(Free_nodes))

idx_Fixed = torch.tensor(Fixed_nodes) - 1
idx_Free = torch.tensor(Free_nodes) - 1 

8


C:\Users\Administrator\AppData\Local\Temp\ipykernel_36680\4248919371.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  idx_Fixed = torch.tensor(Fixed_nodes) - 1


In [8]:

def find_connectivity_indices(grid_points, connectivity, free_nodes):
    
    
    grid_points = torch.tensor(grid_points, dtype=torch.float32)
    
    connectivity = torch.tensor(connectivity, dtype=torch.long)
    
    free_nodes = torch.tensor(free_nodes, dtype=torch.long)
    
    result_indices_x = []  
    result_indices_y = []  
    
    prev_x_list = []  
    prev_y_list = []  
    
    for node in free_nodes:
        node_coord = grid_points[node - 1] 
        x = node_coord[0]

        if x in prev_x_list:
            x_index = prev_x_list.index(x)
        else:
            result_indices_x.append([])
            prev_x_list.append(x)
            x_index = len(prev_x_list) - 1
        
        mask = (connectivity == node).any(dim=1)
        candidate_indices = torch.where(mask)[0] # indexing connectivity
        
        for idx in candidate_indices:
            conn = connectivity[idx]
            coord1 = grid_points[conn[0] - 1]
            coord2 = grid_points[conn[1] - 1]

            if coord1[0] == coord2[0] and coord1[0] == x:
                result_indices_x[x_index].append(idx.item())  
    
    for node in free_nodes:
        node_coord = grid_points[node - 1]  
        y = node_coord[1]
        
        if y in prev_y_list:
            y_index = prev_y_list.index(y)
        else:
            result_indices_y.append([])
            prev_y_list.append(y)
            y_index = len(prev_y_list) - 1
        
        mask = (connectivity == node).any(dim=1)
        print(mask)
        candidate_indices = torch.where(mask)[0]
        print(candidate_indices)
        for idx in candidate_indices:
            conn = connectivity[idx]
            coord1 = grid_points[conn[0] - 1]
            coord2 = grid_points[conn[1] - 1]
            
            if coord1[1] == coord2[1] and coord1[1] == y:
                result_indices_y[y_index].append(idx.item()) 

    print(" x unorganized：", result_indices_x)
    print(" y unorganized：", result_indices_y)
    
 ##### Do the clean-up

    max_len_x = max(len(indices) for indices in result_indices_x) if result_indices_x else 0
    max_len_y = max(len(indices) for indices in result_indices_y) if result_indices_y else 0
    
    
    for indices in result_indices_x:
        indices += [-1] * (max_len_x - len(indices))
    for indices in result_indices_y:
        indices += [-1] * (max_len_y - len(indices))
    
    
    result_x = torch.tensor(result_indices_x, dtype=torch.long)
    result_y = torch.tensor(result_indices_y, dtype=torch.long)
    result_x = torch.unique(result_x, dim=1)
    result_y = torch.unique(result_y, dim=1)
    return result_x, result_y

idx_X, idx_Y = find_connectivity_indices(grid_points, connectivity, Free_nodes)
print("x indices：")
print(idx_X,)
print("y indices：")
print(idx_Y)

tensor([ True,  True, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False,  True, False,
        False, False, False, False])
tensor([ 0,  1, 18])
tensor([False,  True,  True, False, False, False, False, False, False, False,
        False, False, False, False, False,  True, False, False, False, False,
        False, False, False, False])
tensor([ 1,  2, 15])
tensor([False, False, False,  True,  True, False, False, False, False, False,
        False, False, False, False, False, False, False, False,  True,  True,
        False, False, False, False])
tensor([ 3,  4, 18, 19])
tensor([False, False, False, False,  True,  True, False, False, False, False,
        False, False, False, False, False,  True,  True, False, False, False,
        False, False, False, False])
tensor([ 4,  5, 15, 16])
tensor([False, False, False, False, False, False,  True,  True, False, False,
        False, False, False, False, False, False, False, F

C:\Users\Administrator\AppData\Local\Temp\ipykernel_36680\270520761.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  grid_points = torch.tensor(grid_points, dtype=torch.float32)


In [9]:
####### Symetry reshaper -- quires even numbers of indices
len_y = idx_Y.size(0)
half_y = len_y // 2
len_x = idx_X.size(0)
half_x = len_x // 2

y_upper = idx_Y[:half_y]  
y_lower = idx_Y[half_y:] 
y_lower = torch.flip(y_lower, dims=[0])
x_upper = idx_X[:half_x]  
x_lower = idx_X[half_x:]  
x_lower = torch.flip(x_lower, dims=[0])


idx_Y = torch.cat((y_upper, y_lower), dim=1)
idx_X = torch.cat((x_upper, x_lower), dim=1)   

print("Recomposed Y mat:")
print(idx_Y)
print("Recomposed X mat:")
print(idx_X)
con = torch.tensor(connectivity)
X_head = con[idx_X[:,0]][:,0] - 1
Y_head = con[idx_Y[:,0]][:,0] - 1
head = torch.cat((X_head, Y_head))  
print(head)

Recomposed Y mat:
tensor([[18, 19, 20, 15, 16, 17]])
Recomposed X mat:
tensor([[ 0,  1,  2,  9, 10, 11],
        [ 3,  4,  5,  6,  7,  8]])
tensor([0, 4, 1])


In [10]:
###### FE part (CPU version)
D_radius = 0.75
D_young_modulus = 10e9 
D_shear_modulus = 0.7e9 
D_poisson_ratio = 0.3
cross_section_angle_a = 0  
cross_section_angle_b = 0  
a_small_number = 1e-10

def rotation(v, k, theta):
    """Rotation of vector v around axis k by angle theta."""
    k = k / torch.norm(k)  # Normalize k
    cross_product = torch.cross(k, v)
    dot_product = torch.dot(k, v)

    # Ensure theta is a tensor
    theta = torch.tensor(theta, dtype=torch.float32) if not isinstance(theta, torch.Tensor) else theta

    v_rotated = v * torch.cos(theta) + cross_product * torch.sin(theta) + k * dot_product * (1 - torch.cos(theta))
    return v_rotated

class Beam:
    def __init__(self, R, node_coordinates, young_modulus=D_young_modulus,
                 shear_modulus=D_shear_modulus, poisson_ratio=D_poisson_ratio, Beta_a=cross_section_angle_a,
                 Beta_b=cross_section_angle_b):
        self.node_coordinates = node_coordinates  # (2, 3) tensor for node coordinates

        # Material and geometry
        self.radius = R
        self.young_modulus = young_modulus
        self.shear_modulus = shear_modulus
        self.poisson_ratio = poisson_ratio

        # Cross-sectional properties
        self.length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])  # Length of the beam
        self.Iy = (torch.pi * self.radius ** 4) / 4 
        self.Iz = self.Iy
        self.A = torch.pi * self.radius ** 2
        self.J = (torch.pi * self.radius ** 4) / 2

        # Stiffness components
        self.S_u = self.young_modulus * self.A / self.length
        self.S_v1a = 12 * self.young_modulus * self.Iy / (self.length ** 3)
        self.S_v1b = 6 * self.young_modulus * self.Iy / (self.length ** 2)
        self.S_v2a = 12 * self.young_modulus * self.Iz / (self.length ** 3)
        self.S_v2b = 6 * self.young_modulus * self.Iz / (self.length ** 2)
        self.S_theta1a = 6 * self.young_modulus * self.Iy / (self.length ** 2)
        self.S_theta1b = 4 * self.young_modulus * self.Iy / self.length
        self.S_theta1c = 2 * self.young_modulus * self.Iy / self.length
        self.S_theta2a = 6 * self.young_modulus * self.Iz / (self.length ** 2)
        self.S_theta2b = 4 * self.young_modulus * self.Iz / self.length
        self.S_theta2c = 2 * self.young_modulus * self.Iz / self.length
        self.S_Tr = self.shear_modulus * self.J / self.length

        # Section rotations at the two ends
        self.Beta_a = Beta_a
        self.Beta_b = Beta_b

    def get_element_stiffness_matrix(self):
        """Element stiffness matrix."""
        K_element = torch.tensor([
            [self.S_u, 0, 0, 0, 0, 0, -self.S_u, 0, 0, 0, 0, 0],
            [0, self.S_v1a, 0, 0, 0, self.S_theta1a, 0, -self.S_v1a, 0, 0, 0, self.S_theta1a],
            [0, 0, self.S_v2a, 0, -self.S_theta2a, 0, 0, 0, -self.S_v2a, 0, -self.S_theta2a, 0],
            [0, 0, 0, self.S_Tr, 0, 0, 0, 0, 0, -self.S_Tr, 0, 0],
            [0, 0, -self.S_v2b, 0, self.S_theta2b, 0, 0, 0, self.S_v2b, 0, self.S_theta2c, 0],
            [0, self.S_v1b, 0, 0, 0, self.S_theta1b, 0, -self.S_v1b, 0, 0, 0, self.S_theta1c],
            [-self.S_u, 0, 0, 0, 0, 0, self.S_u, 0, 0, 0, 0, 0],
            [0, -self.S_v1a, 0, 0, 0, -self.S_theta1a, 0, self.S_v1a, 0, 0, 0, -self.S_theta1a],
            [0, 0, -self.S_v2a, 0, self.S_theta2a, 0, 0, 0, self.S_v2a, 0, self.S_theta2a, 0],
            [0, 0, 0, -self.S_Tr, 0, 0, 0, 0, 0, self.S_Tr, 0, 0],
            [0, 0, -self.S_v2b, 0, self.S_theta2c, 0, 0, 0, self.S_v2b, 0, self.S_theta2b, 0],
            [0, self.S_v1b, 0, 0, 0, self.S_theta1c, 0, -self.S_v1b, 0, 0, 0, self.S_theta1b],
        ], dtype=torch.float32)

        return K_element

    def System_Transform(self):
        """Coordinate transformation matrix."""
        vector_x = self.node_coordinates[1, 0] - self.node_coordinates[0, 0]
        vector_y = self.node_coordinates[1, 1] - self.node_coordinates[0, 1]
        vector_z = self.node_coordinates[1, 2] - self.node_coordinates[0, 2]
        length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])
        
        z_value = torch.clamp(vector_z / length, min=-1 + 1e-6, max=1 - 1e-6)
        ceta = torch.acos(z_value)
        value = vector_x / torch.sqrt(vector_y ** 2 + vector_x ** 2 + a_small_number)
        value = torch.clamp(value, min=-1 + 1e-6, max=1 - 1e-6)
        alpha = torch.acos(value)

        Projection_Z_x = - vector_z / length * torch.sin(alpha)
        Projection_Z_y = - vector_z / length * torch.cos(alpha)
        Projection_Z_z = torch.cos(torch.pi / 2 - ceta)

        V_projection = torch.stack([Projection_Z_x, Projection_Z_y, Projection_Z_z])
        X_axis = torch.stack([vector_x / length, vector_y / length, vector_z / length])
        Z_axis_a = rotation(V_projection, X_axis, self.Beta_a)
        Y_axis_a = rotation(Z_axis_a, X_axis, -torch.pi / 2)
        Z_axis_a = Z_axis_a / torch.norm(Z_axis_a)
        Y_axis_a = Y_axis_a / torch.norm(Y_axis_a)

        lambda_matrix = torch.stack([X_axis, Y_axis_a, Z_axis_a], dim=0)
        matrix_T = torch.zeros((12, 12), dtype=torch.float32)
        for i in range(0, 12, 3):
            matrix_T[i:i + 3, i:i + 3] = lambda_matrix
        return matrix_T

    def nodal_transform(self):
        """Coordinate transformation matrix."""
        vector_x = self.node_coordinates[1, 0] - self.node_coordinates[0, 0]
        vector_y = self.node_coordinates[1, 1] - self.node_coordinates[0, 1]
        vector_z = self.node_coordinates[1, 2] - self.node_coordinates[0, 2]
        length = torch.norm(self.node_coordinates[1] - self.node_coordinates[0])
        
        z_value = torch.clamp(vector_z / length, min=-1 + 1e-6, max=1 - 1e-6)
        ceta = torch.acos(z_value)
        value = vector_x / torch.sqrt(vector_y ** 2 + vector_x ** 2 + a_small_number)
        value = torch.clamp(value, min=-1 + 1e-6, max=1 - 1e-6)
        alpha = torch.acos(value)

        Projection_Z_x = - vector_z / length * torch.sin(alpha)
        Projection_Z_y = - vector_z / length * torch.cos(alpha)
        Projection_Z_z = torch.cos(torch.pi / 2 - ceta)

        V_projection = torch.stack([Projection_Z_x, Projection_Z_y, Projection_Z_z])
        X_axis = torch.stack([vector_x / length, vector_y / length, vector_z / length])
        Z_axis_a = rotation(V_projection, X_axis, self.Beta_a)
        Y_axis_a = rotation(Z_axis_a, X_axis, -torch.pi / 2)
        Z_axis_a = Z_axis_a / torch.norm(Z_axis_a)
        Y_axis_a = Y_axis_a / torch.norm(Y_axis_a)

        lambda_matrix = torch.stack([X_axis, Y_axis_a, Z_axis_a], dim=0)
        return lambda_matrix


def assemble_stiffness_matrix(beams, n_nodes, n_dof_per_node, connectivity):
    """Global stiffness matrix assembly."""
    total_dof = n_nodes * n_dof_per_node  # Total degrees of freedom
    K_global = torch.zeros((total_dof, total_dof), dtype=torch.float32)
    
    for idx, (i, j) in enumerate(connectivity):
        Matrix_T = beams[idx].System_Transform()  # Get transformation matrix
        K_element = torch.matmul(torch.transpose(Matrix_T, 0, 1),
                                 torch.matmul(beams[idx].get_element_stiffness_matrix(), Matrix_T))

        start_idx = (i - 1) * n_dof_per_node
        end_idx = (j - 1) * n_dof_per_node
        K_global[start_idx:start_idx + 6, start_idx:start_idx + 6] += K_element[0:6, 0:6]
        K_global[end_idx:end_idx + 6, end_idx:end_idx + 6] += K_element[6:12, 6:12]
        K_global[start_idx:start_idx + 6, end_idx:end_idx + 6] += K_element[0:6, 6:12]
        K_global[end_idx:end_idx + 6, start_idx:start_idx + 6] += K_element[6:12, 0:6]

    return K_global

def robust_solve(K_global, F, fixed_dof, max_attempts=3):
    attempts = 0
    while attempts < max_attempts:
        reg = 1e-6 * torch.eye(K_global.shape[0])
        reg[fixed_dof, fixed_dof] = 0  
        K_reg = K_global + reg
        
        try:
            displacements = torch.linalg.solve(
                K_reg.to(torch.float64), 
                F.to(torch.float64)
            )
            return displacements.to(K_global.dtype)
            
        except RuntimeError:
            diag = torch.diag(K_global)
            extreme_mask = (diag > 1e12) & (~torch.isin(torch.arange(len(diag)), torch.tensor(fixed_dof)))  
            K_reg[extreme_mask] = 0
            K_reg[:, extreme_mask] = 0
            K_reg[extreme_mask, extreme_mask] = 1e12  
            
            K_reg[fixed_dof, :] = 0
            K_reg[:, fixed_dof] = 0
            K_reg[fixed_dof, fixed_dof] = 1e10  
            
            try:
                displacements, info = torch.linalg.cg(
                    K_reg.to(torch.float64),
                    F.to(torch.float64),
                    maxiter=5000,
                    atol=1e-6
                )
                if info > 0:
                    raise RuntimeError("CG failed")
                return displacements.to(K_global.dtype)
                
            except:
                K_pinv = torch.linalg.pinv(K_reg)
                K_pinv[fixed_dof, :] = 0  
                displacements = K_pinv @ F
                print("Warning: Using pseudo-inverse solution, accuracy may be reduced")
                return displacements
                
        attempts += 1
    
    raise RuntimeError("Failed to solve linear system")


def Strain_E(node_coords, connectivity, fixed_dof, F, force, D_radius = D_radius):
    # Element Assembly
    Beam_lens = []
    beams = []
    for idx, connection in enumerate(connectivity):
        node_1_coords = node_coords[connection[0] - 1]
        node_2_coords = node_coords[connection[1] - 1]
        length = torch.norm(node_2_coords[1] - node_1_coords[0]) 
        if force[idx] == 0:
            R = D_radius
        elif force[idx] > 0: 
            sigma = 50 
            R = math.sqrt(force[idx] / (math.pi * sigma))
        elif force[idx] < 0:            
            sigma = 100   
            R = math.sqrt(abs(force[idx]) / (math.pi * sigma))
        beam = Beam(R, node_coordinates=torch.stack([node_1_coords, node_2_coords]),
                    young_modulus=D_young_modulus,
                    shear_modulus=D_shear_modulus, poisson_ratio=D_poisson_ratio, Beta_a=cross_section_angle_a,
                    Beta_b=cross_section_angle_b)
        beams.append(beam)
        Beam_lens.append(beam.length)
    
    # Stiffness renewal
    K_global = assemble_stiffness_matrix(beams, n_nodes=len(node_coords), n_dof_per_node=6, connectivity=connectivity)
    K_global[fixed_dof, :] = 0
    K_global[:, fixed_dof] = 0
    K_global[fixed_dof, fixed_dof] = 1e10

    displacements = robust_solve(K_global, F, fixed_dof)

    # Compute strain energy
    strain_energy_list = []
    force_list = []
    Local_d = torch.zeros(len(connectivity), 12, dtype=torch.float32)
    for n, (i, j) in enumerate(connectivity):
        matrix_T = beams[n].System_Transform()
        Tep_displacements = torch.cat(
            [displacements[6 * (i - 1):6 * (i - 1) + 6], displacements[6 * (j - 1):6 * (j - 1) + 6]], dim=0)
        Local_d_n = torch.matmul(Tep_displacements, matrix_T.T)
        Local_d[n, :] = Local_d_n.clone()
        K_l = beams[n].get_element_stiffness_matrix()
        strain_energy_list.append(0.5 * torch.matmul(Local_d_n, torch.matmul(K_l, Local_d_n.reshape(-1, 1))))
        force_list.append(torch.matmul(K_l, Local_d_n.reshape(-1, 1)))

    Strain_energy = torch.stack(strain_energy_list)
    forces = torch.stack(force_list)
    lens = torch.stack(Beam_lens)
    return Strain_energy, forces, displacements, beams, lens

In [11]:
# ###### Plot_func
# import plotly.graph_objects as go

# grid_points = torch.tensor(grid_points)

# def plot_fdm(grid_points, new_node_coords, connectivity, Free_nodes=None):
#     """
#     Plot comparison between original grid points and deformed FDM solution
    
#     Parameters:
#     - grid_points: Tensor/array of original node coordinates (Nx3)
#     - new_node_coords: Tensor/array of deformed node coordinates (Nx3)
#     - connectivity: List of node connections (element edges) as pairs [i,j]
#     - Free_nodes: Optional list of free nodes to highlight (default None)
#     """
#     # Convert to numpy if they're tensors
#     if hasattr(grid_points, 'numpy'):
#         grid_points = grid_points.numpy()
#     if hasattr(new_node_coords, 'numpy'):
#         new_node_coords = new_node_coords.numpy()
    
#     x_orig = grid_points[:, 0]
#     y_orig = grid_points[:, 1]
#     z_orig = grid_points[:, 2]

#     x_def = new_node_coords[:, 0]
#     y_def = new_node_coords[:, 1]
#     z_def = new_node_coords[:, 2]

#     fig = go.Figure()

#     # Plot original structure (blue)
#     for i, j in connectivity:
#         # Note: assuming connectivity uses 1-based indexing
#         fig.add_trace(go.Scatter3d(
#             x=[x_orig[i-1], x_orig[j-1]],
#             y=[y_orig[i-1], y_orig[j-1]],
#             z=[z_orig[i-1], z_orig[j-1]],
#             mode='lines',
#             line=dict(color='blue', width=4),
#             name='Original Geometry',
#             showlegend=False  # Only show legend for the first trace
#         ))

#     # Plot deformed structure (red)
#     for i, j in connectivity:
#         fig.add_trace(go.Scatter3d(
#             x=[x_def[i-1], x_def[j-1]],
#             y=[y_def[i-1], y_def[j-1]],
#             z=[z_def[i-1], z_def[j-1]],
#             mode='lines',
#             line=dict(color='red', width=4),
#             name='FDM Solution',
#             showlegend=False
#         ))

#     # Add legend entries (one for each type)
#     fig.add_trace(go.Scatter3d(
#         x=[None], y=[None], z=[None],
#         mode='lines',
#         line=dict(color='blue', width=4),
#         name='Original Geometry'
#     ))
    
#     fig.add_trace(go.Scatter3d(
#         x=[None], y=[None], z=[None],
#         mode='lines',
#         line=dict(color='red', width=4),
#         name='FDM Solution'
#     ))

#     # Highlight free nodes if provided
#     if Free_nodes is not None:
#         for node in Free_nodes:
#             fig.add_trace(go.Scatter3d(
#                 x=[x_def[node-1]],
#                 y=[y_def[node-1]],
#                 z=[z_def[node-1]],
#                 mode='markers+text',
#                 marker=dict(size=5, color='green'),
#                 text=[f'Node {node}'],
#                 textposition='top center',
#                 name=f'Free Node {node}'
#             ))

#     fig.update_layout(
#         scene=dict(
#             xaxis_title='X',
#             yaxis_title='Y',
#             zaxis_title='Z',
#             aspectmode='data'
#         ),
#         title='Original Geometry vs. FDM Solution',
#         legend=dict(
#             yanchor="top",
#             y=0.99,
#             xanchor="left",
#             x=0.01
#         )
#     )
    
#     return fig

In [12]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def plot_fdm(grid_points, new_node_coords, connectivity, head, q, Free_nodes=None):
    """
    Plot comparison between original grid points and deformed FDM solution using matplotlib
    
    Parameters:
    - grid_points: Tensor/array of original node coordinates (Nx3)
    - new_node_coords: Tensor/array of deformed node coordinates (Nx3)
    - connectivity: List of node connections (element edges) as pairs [i,j]
    - head: Indices of 8 special points in grid_points (length-8 vector)
    - q: Values corresponding to the 8 points (length-8 vector)
    - Free_nodes: Optional list of free nodes to highlight (default None)
    """
    # Convert to numpy if they're tensors
    if hasattr(grid_points, 'numpy'):
        grid_points = grid_points.numpy()
    if hasattr(new_node_coords, 'numpy'):
        new_node_coords = new_node_coords.numpy()
    if hasattr(head, 'numpy'):
        head = head.numpy()
    if hasattr(q, 'numpy'):
        q = q.numpy()
    
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot original structure (blue)
    for i, j in connectivity:
        ax.plot([grid_points[i-1, 0], grid_points[j-1, 0]],
                [grid_points[i-1, 1], grid_points[j-1, 1]],
                [grid_points[i-1, 2], grid_points[j-1, 2]],
                'b--', linewidth=2, alpha=0.7, label='Original' if (i,j) == connectivity[0] else "")
    
    # Plot deformed structure (red)
    for i, j in connectivity:
        ax.plot([new_node_coords[i-1, 0], new_node_coords[j-1, 0]],
                [new_node_coords[i-1, 1], new_node_coords[j-1, 1]],
                [new_node_coords[i-1, 2], new_node_coords[j-1, 2]],
                'r-', linewidth=3, label='FDM Solution' if (i,j) == connectivity[0] else "")
    
    # Highlight free nodes if provided
    if Free_nodes is not None:
        for node in Free_nodes:
            ax.scatter(new_node_coords[node-1, 0], 
                      new_node_coords[node-1, 1], 
                      new_node_coords[node-1, 2],
                      c='g', s=100, marker='o', 
                      label=f'Free Node {node}' if node == Free_nodes[0] else "")
    
    # Plot the 8 special points in grid_points with their q values
    for idx, node in enumerate(head):
        # Convert to 1-based index if needed
        node_idx = node + 1 if np.min(head) == 0 else node
        
        # Plot the point in original grid
        ax.scatter(grid_points[node_idx-1, 0],
                  grid_points[node_idx-1, 1],
                  grid_points[node_idx-1, 2],
                  c='purple', s=150, marker='*')
        
        # Add text label with q value
        ax.text(grid_points[node_idx-1, 0],
               grid_points[node_idx-1, 1],
               grid_points[node_idx-1, 2] + 0.1,  # Slightly above the point
               f'q={q[idx]:.2f}',
               color='black',
               fontsize=10,
               bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))
    
    # Calculate axis ranges with padding
    all_coords = np.vstack([grid_points, new_node_coords])
    min_vals = all_coords.min(axis=0)
    max_vals = all_coords.max(axis=0)
    ranges = max_vals - min_vals
    padding = 0.1 * ranges
    
    ax.set_xlim([min_vals[0]-padding[0], max_vals[0]+padding[0]])
    ax.set_ylim([min_vals[1]-padding[1], max_vals[1]+padding[1]])
    ax.set_zlim([min_vals[2]-padding[2], max_vals[2]+padding[2]])
    
    ax.set_xlabel('Y Axis')
    ax.set_ylabel('X Axis')
    ax.set_zlabel('Z Axis')
    ax.set_title('FDM Geometry')
    
    # Create custom legend without duplicates
    handles, labels = ax.get_legend_handles_labels()
    # Add entry for the special points
    handles.append(plt.Line2D([0], [0], marker='*', color='w', markerfacecolor='purple', markersize=10))
    labels.append('Special Points (q values)')
    
    by_label = dict(zip(labels, handles))  # Remove duplicates
    ax.legend(by_label.values(), by_label.keys(), loc='upper right')
    
    plt.tight_layout()
    return fig

In [13]:
########## Differentiable FDM

# px = torch.zeros(len(f_n), 1, dtype=torch.float32)
px = torch.zeros(len(f_n), 1, dtype=torch.float32)
py = torch.zeros(len(f_n), 1, dtype=torch.float32)
pz = torch.zeros(len(f_n), 1, dtype=torch.float32)

pz[:, 0] = F_value


## Obtaining Matrix D
# Assembly the nodal-branch matrix:
n_elements = len(connectivity)
idx_CF = idx_Fixed
idx_CN = idx_Free
C = torch.zeros(n_elements, n_nodes, dtype=torch.float32)
for n, (i, j) in enumerate(connectivity):
    C[n, i - 1] = 1
    C[n, j - 1] = -1
CF = C[:, idx_CF]
CN = C[:, idx_CN]


# Assembly D
def FDM (Q, C=C, CN=CN, CF=CF, px=px, py=py, pz=pz,Fixed_nodes=Fixed_nodes, Free_nodes=Free_nodes,node_coords=grid_points) :
    Dn = torch.matmul(torch.transpose(CN, 0, 1), torch.matmul(Q, CN))
    DF = torch.matmul(torch.transpose(CN, 0, 1), torch.matmul(Q, CF))
    ## Coordinates solution
    # separate the coordinates:
    fixed_idces = [node - 1 for node in Fixed_nodes]
    xF = node_coords[fixed_idces, 0].unsqueeze(1)
    yF = node_coords[fixed_idces, 1].unsqueeze(1)
    zF = node_coords[fixed_idces, 2].unsqueeze(1)
    xN = torch.matmul(torch.inverse(Dn), (px - torch.matmul(DF, xF)))
    yN = torch.matmul(torch.inverse(Dn), (py - torch.matmul(DF, yF)))
    zN = torch.matmul(torch.inverse(Dn), (pz - torch.matmul(DF, zF)))
    ## Coordinates renewal: 
    new_node_coords = node_coords.clone()
    for idx, node in enumerate(Free_nodes):
        node_idx = node - 1
        new_node_coords[node_idx, 0] = xN[idx]
        new_node_coords[node_idx, 1] = yN[idx]
        new_node_coords[node_idx, 2] = zN[idx]
#     print(new_node_coords[:,0])
    return new_node_coords





####### BCs
fixed_dof = []
for node in Fixed_nodes:
    fixed_dof.extend([(node - 1) * 6 + i for i in range(6)])
    

In [14]:
global new_node_coords
global q_vec
new_node_coords = None
q_vec = None

def interactive_fdm_plot(**q_values):
    global q_vec
    global new_node_coords 
    # Convert slider values to tensor q
    q = torch.tensor([q_values[f'q_{i}'] for i in range(n_elem)], 
                     dtype=torch.float32)
    
    q_vec = torch.zeros(n_elements)

    for i in range(len(idx_X)):
        q_vec[idx_X[i,:]] = q[i] 
    for j in range(len(idx_Y)):
        q_vec[idx_Y[j,:]] = q[j+len(idx_X)]
    
    q_vec = q_vec * 1 / r
    Q = torch.diag(q_vec)  
    
    
    new_node_coords = FDM(Q)
    
    Z = new_node_coords[:,2]
    
    print('height', max(Z))
    
    # Plot the result
    fig = plot_fdm(grid_points, new_node_coords, connectivity, head, q)
    fig.show()
    

# Create sliders for each element
n_elem = len(idx_X) + len(idx_Y)  


sliders = {}
for i in range(n_elem):
    sliders[f'q_{i}'] = widgets.FloatSlider(
        value=1.0,
        min=0,
        max=6,
        step=0.1,
        description=f'Element {i}',
        continuous_update=True,  # 实时更新
        readout=True,
        orientation='horizontal',
        layout=widgets.Layout(width='500px')  # 调整滑块宽度
    )

# 使用VBox组织滑块
slider_box = widgets.VBox([sliders[f'q_{i}'] for i in range(n_elem)])

# 创建交互式控件
interactive_plot = widgets.interactive(
    interactive_fdm_plot,
    **sliders
)

# 添加标题和整体布局
title = widgets.HTML("<h3>FDM Interactive Parameter Control</h3>")

display(widgets.VBox([title, interactive_plot]))

# 添加一个重置按钮
def reset_sliders(b):
    for i in range(n_elem):
        sliders[f'q_{i}'].value = 5.44
reset_button = widgets.Button(description="Reset All")
reset_button.on_click(reset_sliders)
display(reset_button)


Button(description='Reset All', style=ButtonStyle())

In [15]:
# ###########. Embbeding FE solver :
# F_fe_c = torch.zeros(total_dof, dtype=torch.float32)
# load_value_c = torch.tensor([1] * len(Free_nodes)) * 1000 # The force value/direction
# f_type_c = [0] * len(Free_nodes)  # The force type
# for idx, i in enumerate(f_n):
#     F_fe_c[6 * (i - 1) + f_type_c[idx]] = load_value_c[idx]  # unit: KN / KN*m

# force_vec = torch.zeros(len(connectivity), dtype=torch.float32)

# q_v = torch.tensor([
#         1.5,
#         1.449279546737671,
#         1.456311821937561,
#         1.4528206586837769,
#         1.4346059560775757,
#         0.0,
#         0.0,
#         0.0
#       ]) 
# q_vec = torch.zeros(n_elements)
# for i in range(len(idx_X)):
#     q_vec[idx_X[i,:]] = q_v[i] 
# for j in range(len(idx_Y)):
#     q_vec[idx_Y[j,:]] = q_v[j+len(idx_X)] 
# q_vec = q_vec * 1 / r 
# Q_t = torch.diag(q_vec) 
# new_node_coords = FDM(Q_t)

# _, _, _,_, Beam_lens = Strain_E(new_node_coords, connectivity, fixed_dof, F_fe_c, force_vec)

# force_vec = - q_vec * Beam_lens * 0.7484157085418701

# _, _, _,Beams, Beam_lens = Strain_E(new_node_coords, connectivity, fixed_dof, F_fe_c, force_vec)

# Volume_list =  []
# for n in range(len(connectivity)):
#     Volume_list.append(Beams[n].A * Beam_lens[n])
# Volume = sum(Volume_list)

# F_fe = torch.zeros(total_dof, dtype=torch.float32)
# load_value = torch.tensor([-1] * len(Free_nodes)) * 1000 # The force value/direction
# f_type = [2] * len(Free_nodes)  # The force type
# for idx, i in enumerate(f_n):
#     F_fe[6 * (i - 1) + f_type[idx]] = load_value[idx]  # unit: KN / KN*m

# Strain_energy, forces, displacements, _, Beam_lens = Strain_E(new_node_coords, connectivity, fixed_dof, F_fe, force_vec)

# SE = torch.sum(Strain_energy) 
# force = abs(forces[:, 0, 0])
# LP = torch.dot(force , Beam_lens)
# Lp_q = torch.dot(abs(q_vec * Beam_lens) , Beam_lens)
  


# print('SE', SE)
# print('LP', LP)
# print('Lp_q', Lp_q)
# print('Volume', Volume)
# Height = max(new_node_coords[:,2])
# print(Height)

In [16]:
# import plotly.graph_objects as go
# x_orig = grid_points[:, 0].cpu().detach().numpy()
# y_orig = grid_points[:, 1].cpu().detach().numpy()
# z_orig = grid_points[:, 2].cpu().detach().numpy()

# x_fdm = new_node_coords[:, 0].cpu().detach().numpy()
# y_fdm = new_node_coords[:, 1].cpu().detach().numpy()
# z_fdm = new_node_coords[:, 2].cpu().detach().numpy()

# fig = go.Figure()

# for connection in connectivity:
#     i, j = connection
#     fig.add_trace(go.Scatter3d(
#         x=[x_orig[i-1], x_orig[j-1]],
#         y=[y_orig[i-1], y_orig[j-1]],
#         z=[z_orig[i-1], z_orig[j-1]],
#         mode='lines',
#         line=dict(color='blue', width=1),
#         name='Grid',
#         showlegend=False
#     ))

# for connection in connectivity:
#     i, j = connection
#     fig.add_trace(go.Scatter3d(
#         x=[x_fdm[i-1], x_fdm[j-1]],
#         y=[y_fdm[i-1], y_fdm[j-1]],
#         z=[z_fdm[i-1], z_fdm[j-1]],
#         mode='lines',
#         line=dict(color='red', width=4),
#         name='FDM solution',
#         showlegend=False
#     ))


# for node in Fixed_nodes:
#     fig.add_trace(go.Scatter3d(
#         x=[x_fdm[node-1]],
#         y=[y_fdm[node-1]],
#         z=[z_fdm[node-1]],
#         mode='markers+text',
#         marker=dict(size=5, color='black'),
#         name=f'Fixed Node {node}',
#         showlegend=False
#     ))
    

# force_traces = []
# force_np = force.cpu().detach().numpy() 
# for idx, connection in enumerate(connectivity):
#     i, j = connection
#     mid_x = (x_fdm[i-1] + x_fdm[j-1]) / 2
#     mid_y = (y_fdm[i-1] + y_fdm[j-1]) / 2
#     mid_z = (z_fdm[i-1] + z_fdm[j-1]) / 2
#     trace = go.Scatter3d(
#         x=[mid_x],
#         y=[mid_y],
#         z=[mid_z],
#         mode='markers+text',
#         marker=dict(size=1, color='green'),
#         text=[f"{force_np[idx]:.0f}"],
#         textposition='top center',
#         textfont=dict(size=8),
#         name=f'Force {idx+1}',
#         visible=True
#     )
#     force_traces.append(trace)
#     fig.add_trace(trace)

# fig.update_layout(
#     updatemenus=[
#         dict(
#             type="buttons",
#             direction="right",
#             x=0.1,
#             y=1.1,
#             buttons=[
#                 dict(
#                     label="✅ Show forces",
#                     method="update",
#                     args=[{"visible": [True] * len(fig.data)}],
#                 ),
#                 dict(
#                     label="❌ Hide forces",
#                     method="update",
#                     args=[{"visible": [True] * (len(fig.data) - len(force_traces)) + [False] * len(force_traces)}],
#                 )
#             ]
#         )
#     ],
#     scene=dict(
#         xaxis=dict(
#             showbackground=False,
#             showgrid=False,
#             showline=False,
#             showticklabels=False,
#             title=''
#         ),
#         yaxis=dict(
#             showbackground=False,
#             showgrid=False,
#             showline=False,
#             showticklabels=False,
#             title=''
#         ),
#         zaxis=dict(
#             showbackground=False,
#             showgrid=False,
#             showline=False,
#             showticklabels=False,
#             title=''
#         ),
#         aspectmode='data'
#     ),
# #     title='Test with Lateral load (Y)',
#      title='Test with Gravity load',
#     annotations=[
#         dict(
#             x=0.05,  # X position (0-1, left to right)
#             y=0.95,  # Y position (0-1, bottom to top)
#             xref="paper",
#             yref="paper",
#             text=f"Strain energy = {SE:.4f}, Volume = {Volume:.4f}, Height = {Height:.4f}, Load_path={LP:.4f}",
#             showarrow=False,
#             font=dict(
#                 size=14,
#                 color="black"
#             ),
#             bgcolor="white",
#             bordercolor="black",
#             borderwidth=1,
#             borderpad=4
#         )
#     ]
# )

# fig.show()
# fig.write_html("E_strain Test(Z).html")
